# Trout Age6 SimCLR: 4+/5+ Combined

This notebook trains SimCLR for a 6-class trout label task:

- `0+`
- `1+`
- `2+`
- `3+`
- `4+/5+` = original labels `4` and `5`
- `6 / bad` = original label `6`, not readable, regenerated, broken, or otherwise unusable scale

Input information:
- scale images only;
- no fish length;
- no fish weight.

Motivation: original `5+` has very few samples, so it is combined with `4+` for a more stable older-age class.

## 1. Setup

Run `trout_new_dataset_eda.ipynb` first. This notebook reads `eda_outputs/master_table_new.csv`.

In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

ROOT_DIR = Path(os.environ.get("TROUT_ROOT_DIR", "/home/jlc3q/data/Trout"))
CODE_DIR = Path(os.environ.get("TROUT_CODE_DIR", str(ROOT_DIR / "code_new")))
EDA_OUTPUT_DIR = CODE_DIR / "eda_outputs"
MODEL_OUTPUT_DIR = CODE_DIR / "model_outputs"
CHECKPOINT_DIR = MODEL_OUTPUT_DIR / "checkpoints"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = EDA_OUTPUT_DIR / "master_table_new.csv"

SEED = 100
TEST_SIZE = 0.2
CLASS_NAMES = ["0+", "1+", "2+", "3+", "4+/5+", "6 / bad"]
LABELS = list(range(6))

SIMCLR_EPOCHS = 10
CLASSIFIER_EPOCHS = 10
SIMCLR_BATCH_SIZE = 128
CLASSIFIER_BATCH_SIZE = 64
SIMCLR_LR = 3e-4
CLASSIFIER_LR = 1e-4
TEMPERATURE = 0.2

# Fair default: SimCLR uses only labeled train fish images.
# Set True if you intentionally want to use all images from train fish, including unlabeled scales, for self-supervised pretraining.
USE_UNLABELED_TRAIN_FISH_FOR_SIMCLR = False

random.seed(SEED)
np.random.seed(SEED)

print("ROOT_DIR:", ROOT_DIR)
print("CODE_DIR:", CODE_DIR)
print("MASTER_CSV exists:", MASTER_CSV.exists())

## 2. Load Full Label Table

This uses rows where `label` is not null. Original labels `4` and `5` are combined, and original label `6` is kept as the bad/readability class.

In [ ]:
if not MASTER_CSV.exists():
    raise FileNotFoundError(f"Missing {MASTER_CSV}. Run trout_new_dataset_eda.ipynb first.")

master_df = pd.read_csv(MASTER_CSV)
required_cols = {"path", "scale_id", "fish_key", "label"}
missing_cols = required_cols - set(master_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

model_df = master_df[master_df["label"].notna()].copy()
def map_age6(label):
    label = int(label)
    if label <= 3:
        return label
    if label in [4, 5]:
        return 4
    if label == 6:
        return 5
    raise ValueError(f"Unexpected label: {label}")

model_df["original_label"] = model_df["label"].astype(int)
model_df["age6"] = model_df["original_label"].map(map_age6).astype(int)
model_df["path_exists"] = model_df["path"].map(lambda p: Path(str(p)).exists())
model_df = model_df[model_df["path_exists"]].copy().reset_index(drop=True)

print("model_df:", model_df.shape)
print("unique fish:", model_df["fish_key"].nunique())
print("original label counts:")
display(model_df["original_label"].value_counts().reindex(range(7), fill_value=0).rename_axis("original_label").reset_index(name="count"))
print("age6 counts:")
display(model_df["age6"].value_counts().reindex(LABELS, fill_value=0).rename_axis("age6").reset_index(name="count"))
display(model_df[["scale_id", "fish_key", "original_label", "age6", "path"]].head())

## 3. Fish-Level Split

This searches deterministic random seeds until train and test both contain all six target classes. It still enforces zero fish overlap.

In [ ]:
def make_group_split_with_all_classes(df: pd.DataFrame, max_seed_tries: int = 500):
    best = None
    for seed in range(SEED, SEED + max_seed_tries):
        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        train_idx, test_idx = next(splitter.split(df, df["age6"], groups=df["fish_key"]))
        train_df_candidate = df.iloc[train_idx].copy().reset_index(drop=True)
        test_df_candidate = df.iloc[test_idx].copy().reset_index(drop=True)
        train_counts = train_df_candidate["age6"].value_counts().reindex(LABELS, fill_value=0)
        test_counts = test_df_candidate["age6"].value_counts().reindex(LABELS, fill_value=0)
        min_test_count = int(test_counts.min())
        score = min_test_count
        if best is None or score > best[0]:
            best = (score, seed, train_df_candidate, test_df_candidate, train_counts, test_counts)
        if (train_counts > 0).all() and (test_counts > 0).all():
            return seed, train_df_candidate, test_df_candidate, train_counts, test_counts
    print("WARNING: could not find a split where every class appears in both train and test.")
    _, seed, train_df_candidate, test_df_candidate, train_counts, test_counts = best
    return seed, train_df_candidate, test_df_candidate, train_counts, test_counts

split_seed, train_df, test_df, train_counts, test_counts = make_group_split_with_all_classes(model_df)
fish_overlap = sorted(set(train_df["fish_key"]) & set(test_df["fish_key"]))

print("split_seed:", split_seed)
print("train:", train_df.shape)
display(train_counts.rename_axis("age6").reset_index(name="train_count"))
print("test:", test_df.shape)
display(test_counts.rename_axis("age6").reset_index(name="test_count"))
print("Fish overlap:", len(fish_overlap))
assert len(fish_overlap) == 0

## 4. Torch Setup

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision.transforms as T
    import torchvision.models as tv_models
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    from tqdm.auto import tqdm
except ImportError as exc:
    raise ImportError(
        "This notebook needs torch, torchvision, pillow, and tqdm in the active Jupyter kernel."
    ) from exc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = device.type == "cuda"
NUM_WORKERS = 0
PIN_MEMORY = USE_CUDA


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("device:", device)
print("num_workers:", NUM_WORKERS)

## 5. Datasets, Transforms, and Loss

In [ ]:
simclr_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomResizedCrop(224, scale=(0.65, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.25, contrast=0.25),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_classifier_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SimCLRDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
        return self.transform(img), self.transform(img)

class ImageLabelDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.y = self.df["age6"].astype(int).to_numpy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
        return self.transform(img), int(self.y[idx])


def make_resnet18_backbone(imagenet_init: bool = True):
    try:
        weights = tv_models.ResNet18_Weights.DEFAULT if imagenet_init else None
        model = tv_models.resnet18(weights=weights)
    except AttributeError:
        model = tv_models.resnet18(pretrained=imagenet_init)
    model.fc = nn.Identity()
    return model

class NTXentLoss(nn.Module):
    def __init__(self, temperature: float = TEMPERATURE):
        super().__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss(reduction="sum")

    def forward(self, z1, z2):
        batch_size = z1.size(0)
        z = torch.cat((z1, z2), dim=0)
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

        positive = torch.cat([
            torch.diag(sim, batch_size),
            torch.diag(sim, -batch_size),
        ], dim=0).unsqueeze(1)

        mask = torch.ones((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=z.device)
        mask.fill_diagonal_(False)
        for i in range(batch_size):
            mask[i, batch_size + i] = False
            mask[batch_size + i, i] = False

        negative = sim[mask].view(2 * batch_size, -1)
        logits = torch.cat([positive, negative], dim=1) / self.temperature
        labels = torch.zeros(2 * batch_size, dtype=torch.long, device=z.device)
        return self.criterion(logits, labels) / (2 * batch_size)

print("components ready")

## 6. SimCLR Pretraining

In [ ]:
if USE_UNLABELED_TRAIN_FISH_FOR_SIMCLR:
    train_fish = set(train_df["fish_key"])
    simclr_df = master_df[master_df["fish_key"].isin(train_fish)].copy()
    simclr_df = simclr_df[simclr_df["path"].map(lambda p: Path(str(p)).exists())].reset_index(drop=True)
else:
    simclr_df = train_df.copy()

print("simclr_df:", simclr_df.shape)
print("unique fish for SimCLR:", simclr_df["fish_key"].nunique())
print("uses unlabeled train fish images:", USE_UNLABELED_TRAIN_FISH_FOR_SIMCLR)

In [ ]:
def train_simclr_backbone(dataframe: pd.DataFrame, epochs: int = SIMCLR_EPOCHS):
    set_seed()
    dataset = SimCLRDataset(dataframe, simclr_transform)
    loader = DataLoader(
        dataset,
        batch_size=SIMCLR_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    backbone = make_resnet18_backbone(imagenet_init=True).to(device)
    projection_head = nn.Sequential(
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Linear(128, 128),
    ).to(device)

    optimizer = torch.optim.AdamW(
        list(backbone.parameters()) + list(projection_head.parameters()),
        lr=SIMCLR_LR,
        weight_decay=1e-4,
    )
    criterion = NTXentLoss(temperature=TEMPERATURE)

    backbone.train()
    projection_head.train()
    for epoch in range(epochs):
        losses = []
        for x1, x2 in tqdm(loader, desc=f"SimCLR {epoch + 1}/{epochs}"):
            x1 = x1.to(device)
            x2 = x2.to(device)
            z1 = projection_head(backbone(x1))
            z2 = projection_head(backbone(x2))
            loss = criterion(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")

    ckpt_path = CHECKPOINT_DIR / "age6_45combined_simclr_resnet18_backbone.pt"
    torch.save({
        "backbone_state_dict": backbone.state_dict(),
        "simclr_epochs": epochs,
        "simclr_batch_size": SIMCLR_BATCH_SIZE,
        "temperature": TEMPERATURE,
        "use_unlabeled_train_fish": USE_UNLABELED_TRAIN_FISH_FOR_SIMCLR,
    }, ckpt_path)
    print("saved:", ckpt_path)
    return backbone

simclr_backbone = train_simclr_backbone(simclr_df, epochs=SIMCLR_EPOCHS)

## 7. Age6 Classifier

In [ ]:
class Age6Classifier(nn.Module):
    def __init__(self, backbone: nn.Module, n_classes: int = 6):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, image):
        return self.head(self.backbone(image))


def class_weight_tensor(y: pd.Series) -> torch.Tensor:
    counts = y.value_counts().reindex(LABELS, fill_value=0).astype(float)
    weights = counts.sum() / (len(counts) * counts.clip(lower=1))
    return torch.tensor(weights.to_numpy(), dtype=torch.float32, device=device)

print("class weights:")
display(pd.DataFrame({
    "label": LABELS,
    "class_name": CLASS_NAMES,
    "train_count": train_counts.values,
    "weight": class_weight_tensor(train_df["age6"]).detach().cpu().numpy(),
}))

In [ ]:
def train_age6_classifier(backbone: nn.Module, epochs: int = CLASSIFIER_EPOCHS):
    set_seed()
    train_dataset = ImageLabelDataset(train_df, train_classifier_transform)
    test_dataset = ImageLabelDataset(test_df, eval_transform)
    train_loader = DataLoader(train_dataset, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_dataset, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = Age6Classifier(backbone).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(train_df["age6"]))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CLASSIFIER_LR, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        losses = []
        for images, y in tqdm(train_loader, desc=f"Age6 classifier {epoch + 1}/{epochs}"):
            images = images.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")

    ckpt_path = CHECKPOINT_DIR / "age6_45combined_simclr_resnet18_classifier.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "classifier_epochs": epochs,
        "class_names": CLASS_NAMES,
        "split_seed": split_seed,
    }, ckpt_path)
    print("saved:", ckpt_path)
    return model, test_loader

age6_model, age6_test_loader = train_age6_classifier(simclr_backbone, epochs=CLASSIFIER_EPOCHS)

## 8. Evaluation

In [ ]:
def evaluate_predictions(name: str, y_true, y_pred) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
        "support": int(len(y_true)),
    }

    print("\n===", name, "===")
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("Balanced accuracy:", round(metrics["balanced_accuracy"], 4))
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print(classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))
    return metrics

@torch.no_grad()
def evaluate_torch_model(model: nn.Module, loader: DataLoader, name: str):
    model.eval()
    y_true = []
    y_pred = []
    for images, y in tqdm(loader, desc=f"evaluate {name}"):
        logits = model(images.to(device))
        pred = logits.argmax(dim=1).detach().cpu().numpy()
        y_pred.extend(pred.tolist())
        y_true.extend(y.numpy().tolist())
    metrics = evaluate_predictions(name, y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_path = MODEL_OUTPUT_DIR / f"confusion_matrix_{name}.csv"
    cm_df.to_csv(cm_path)
    print("saved:", cm_path)
    display(cm_df)
    return metrics, np.asarray(y_true), np.asarray(y_pred), cm_df

age6_metrics, age6_y_true, age6_y_pred, age6_cm = evaluate_torch_model(age6_model, age6_test_loader, "simclr_resnet18_age6_45combined")

## 9. Save Results

In [ ]:
results_df = pd.DataFrame([age6_metrics])
results_path = MODEL_OUTPUT_DIR / "age6_45combined_simclr_results.csv"
results_df.to_csv(results_path, index=False)
print("saved:", results_path)
display(results_df)

predictions_df = test_df[["scale_id", "fish_key", "path", "age6"]].copy()
predictions_df["pred_age6"] = age6_y_pred
predictions_df["correct"] = predictions_df["age6"].eq(predictions_df["pred_age6"])
predictions_path = MODEL_OUTPUT_DIR / "age6_45combined_simclr_test_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)
print("saved:", predictions_path)
display(predictions_df.head())

In [ ]:
summary = {
    "n_rows": int(len(model_df)),
    "n_train_rows": int(len(train_df)),
    "n_test_rows": int(len(test_df)),
    "n_unique_fish": int(model_df["fish_key"].nunique()),
    "n_train_fish": int(train_df["fish_key"].nunique()),
    "n_test_fish": int(test_df["fish_key"].nunique()),
    "fish_overlap": int(len(set(train_df["fish_key"]) & set(test_df["fish_key"]))),
    "split_seed": int(split_seed),
    "simclr_epochs": SIMCLR_EPOCHS,
    "classifier_epochs": CLASSIFIER_EPOCHS,
    "simclr_batch_size": SIMCLR_BATCH_SIZE,
    "classifier_batch_size": CLASSIFIER_BATCH_SIZE,
    "temperature": TEMPERATURE,
    "use_unlabeled_train_fish_for_simclr": USE_UNLABELED_TRAIN_FISH_FOR_SIMCLR,
    "uses_length_weight": False,
}
summary_path = MODEL_OUTPUT_DIR / "age6_45combined_simclr_run_summary.json"
summary_path.write_text(__import__("json").dumps(summary, indent=2))
print("saved:", summary_path)
summary

## 10. Error Inspection Helpers

Use these after evaluation to inspect difficult cases, especially `5+` and confusion between readable age classes and bad scales.

In [ ]:
errors_df = predictions_df[~predictions_df["correct"]].copy()
print("errors:", errors_df.shape)
print("errors by true/pred:")
display(errors_df.groupby(["age6", "pred_age6"]).size().reset_index(name="n").sort_values("n", ascending=False))

def show_error_group(true_label: int, pred_label: int, n: int = 12):
    subset = errors_df[(errors_df["age6"].eq(true_label)) & (errors_df["pred_age6"].eq(pred_label))].head(n)
    display(subset)
    return subset

# Example:
# age6 class index 4 means original 4+/5+; class index 5 means original 6/bad.
# show_error_group(4, 5)
# show_error_group(2, 3)